In [1]:
######################################
## IMPORTS AND LIBRARY DEPENDENCIES ##
######################################

## Imports
import os
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from f_dataset_augmented import ImageDataset
from f_model_encoder import ResNet18FeatureExtractor
from f_model_encoder import ResNet50FeatureExtractor
from f_model_encoder import ViTFeatureExtractor
from f_model_unet import UNetFeatureExtractor
from f_model_transunet import UNetTransformerFeatureExtractor

#
###

In [2]:
#############################
## SCRIPT CONFIGURATIONS   ##
#############################

## Set device
device = 'mps'

## GLOBAL VARIABLES
FOLDER_NAME = 'icgcc'
MODEL_NAME = 'model_vit_icgcc_raw_FTL_0'
# MODEL_NAME = 'model_resnet50_icgcc_raw_FTL_0'
# MODEL_NAME = 'model_resnet50_tcgaoct_raw_FTL_0'
FILE_FORMAT = '_raw.png'

## Define paths and parameters
dataset_path = f"/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/{FOLDER_NAME}"
batch_size = 128
RESIZE = 224
pto = f"/Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/{FOLDER_NAME}/{MODEL_NAME}/"
output_embeddings_path = pto + 'embeddings.npy'
output_identifiers_path = pto + 'image_identifiers.csv'

## Path to the saved weights file (replace with your actual path)
weights_path = f"models/{MODEL_NAME}.pth"

## Check if output folder exists
if os.path.exists(pto) == False:
    os.mkdir(pto)

#
###

In [3]:
##########################
## LOAD MODEL AND DATA  ##
##########################

## Initialize the ResNet18 model
# model = ResNet18FeatureExtractor(pretrained=True).to(device)
# model = ResNet50FeatureExtractor(pretrained=True).to(device)
model = ViTFeatureExtractor(pretrained=True).to(device)
# model = UNetTransformerFeatureExtractor(n_classes=3).to(device)
# model = UNetFeatureExtractor(n_classes=3).to(device)

if os.path.exists(weights_path):
    ## Load the state dictionary (weights)
    state_dict = torch.load(weights_path, map_location=device)
    ## Load the weights into the model
    model.load_state_dict(state_dict)
    print('Loaded weights')
model.eval()  # Set model to evaluation mode

## Load the dataset
dataset = ImageDataset(folder_path=dataset_path, augment=False, resize=RESIZE, file_format=FILE_FORMAT)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

#
###

In [4]:
##################################
## GENERATE AND SAVE EMBEDDINGS ##
##################################

## Initialize a list to hold all embeddings
embeddings_list = []
identifiers_list = []

## Generate embeddings
k = 0
with torch.no_grad():  ## Disable gradient computation
    for images, identifiers in data_loader:

        print(f"{k+1}/{len(data_loader)}",end='\r')

        ##
        images = images.to(device)
        
        ## Forward pass through the model
        embeddings = model(images)

        ## Convert embeddings to CPU and numpy format, then append to list
        embeddings_list.append(embeddings.cpu().numpy())
        
        ## Append identifiers directly from the dataset
        identifiers_list.extend(identifiers)

        ## Iterator
        k += 1

## Concatenate all embeddings into a single array
embeddings_array = np.concatenate(embeddings_list, axis=0)

## Save embeddings to a .npy file
np.save(output_embeddings_path, embeddings_array)
print(f'Embeddings saved to {output_embeddings_path}')

## Save identifiers to a .csv file
identifiers_df = pd.DataFrame({'filename': identifiers_list})
identifiers_df.to_csv(output_identifiers_path, index=False)
print(f'Image identifiers saved to {output_identifiers_path}')

#
###

Embeddings saved to /Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/icgcc/model_vit_icgcc_raw_FTL_0/embeddings.npy
Image identifiers saved to /Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/icgcc/model_vit_icgcc_raw_FTL_0/image_identifiers.csv


In [5]:
#############################
## SCRIPT CONFIGURATIONS   ##
#############################

## Set device
device = 'mps'

## GLOBAL VARIABLES
FOLDER_NAME = 'tcga-oct'
MODEL_NAME = 'model_vit_tcgaoct_raw_FTL_0'
FILE_FORMAT = '_raw.png'

## Define paths and parameters
dataset_path = f"/Volumes/Elements/BDI/projects/GSG/outputs/o1_extracted_glands/{FOLDER_NAME}"
batch_size = 128
RESIZE = 224
pto = f"/Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/{FOLDER_NAME}/{MODEL_NAME}/"
output_embeddings_path = pto + 'embeddings.npy'
output_identifiers_path = pto + 'image_identifiers.csv'

## Path to the saved weights file (replace with your actual path)
weights_path = f"models/{MODEL_NAME}.pth"

## Check if output folder exists
if os.path.exists(pto) == False:
    os.mkdir(pto)

#
###

##########################
## LOAD MODEL AND DATA  ##
##########################

## Initialize the ResNet18 model
# model = ResNet18FeatureExtractor(pretrained=True).to(device)
# model = ResNet50FeatureExtractor(pretrained=True).to(device)
model = ViTFeatureExtractor(pretrained=True).to(device)
# model = UNetTransformerFeatureExtractor(n_classes=3).to(device)
# model = UNetFeatureExtractor(n_classes=3).to(device)

if os.path.exists(weights_path):
    ## Load the state dictionary (weights)
    state_dict = torch.load(weights_path, map_location=device)
    ## Load the weights into the model
    model.load_state_dict(state_dict)
    print('Loaded weights')
model.eval()  # Set model to evaluation mode

## Load the dataset
dataset = ImageDataset(folder_path=dataset_path, augment=False, resize=RESIZE, file_format=FILE_FORMAT)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

#
###

##################################
## GENERATE AND SAVE EMBEDDINGS ##
##################################

## Initialize a list to hold all embeddings
embeddings_list = []
identifiers_list = []

## Generate embeddings
k = 0
with torch.no_grad():  ## Disable gradient computation
    for images, identifiers in data_loader:

        print(f"{k+1}/{len(data_loader)}",end='\r')

        ##
        images = images.to(device)
        
        ## Forward pass through the model
        embeddings = model(images)

        ## Convert embeddings to CPU and numpy format, then append to list
        embeddings_list.append(embeddings.cpu().numpy())
        
        ## Append identifiers directly from the dataset
        identifiers_list.extend(identifiers)

        ## Iterator
        k += 1

## Concatenate all embeddings into a single array
embeddings_array = np.concatenate(embeddings_list, axis=0)

## Save embeddings to a .npy file
np.save(output_embeddings_path, embeddings_array)
print(f'Embeddings saved to {output_embeddings_path}')

## Save identifiers to a .csv file
identifiers_df = pd.DataFrame({'filename': identifiers_list})
identifiers_df.to_csv(output_identifiers_path, index=False)
print(f'Image identifiers saved to {output_identifiers_path}')

#
###

Embeddings saved to /Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/tcga-oct/model_vit_tcgaoct_raw_FTL_0/embeddings.npy
Image identifiers saved to /Volumes/Elements/BDI/projects/GSG/outputs/o2_embeddings/tcga-oct/model_vit_tcgaoct_raw_FTL_0/image_identifiers.csv
